# Défi quotidien : Comparaison Attention mono-tête / multi-têtes / Transformer pré-entraîné

Ce notebook traite l'intégralité du défi sur l'attention dans les architectures de type
Transformer, à partir du jeu de données **NLI multilingue** (`Contradictory, My Dear Watson`,
3 classes : 0 = entailment, 1 = neutral, 2 = contradiction).

Plan :
1. Chargement et exploration des données
2. Préparation des données (tokenisation maison + tokenizer pré-entraîné)
3. Attention mono-tête (Scaled Dot-Product Attention)
4. Attention multi-têtes (avec dropout et connexion résiduelle)
5. Pile d'encodeurs personnalisée + boucle d'entraînement (optionnel, implémenté)
6. Baseline : fine-tuning d'un Transformer pré-entraîné (DistilBERT multilingue)
7. Comparaison encodeur personnalisé vs Transformer pré-entraîné
8. Visualisation des cartes d'attention
9. Réflexion comparative (mono-tête / multi-têtes / cross-attention, compromis)
10. Liste de contrôle des livrables

> ⚠️ Recommandé : exécuter sur GPU (Colab GPU runtime) pour la section 6 (fine-tuning).


## 0. Imports et configuration

In [ ]:
# Dépendances (décommentez si besoin sur votre environnement)
# !pip install -q transformers torch scikit-learn pandas matplotlib seaborn

import os, math, json, random, zipfile
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader

from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, classification_report

SEED = 42
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Device:", DEVICE)


## 1. Chargement et exploration des données

Le jeu de données est fourni sous forme d'archive zip contenant `train.csv.zip`,
`test.csv.zip` et `sample_submission.csv`. Adaptez `ZIP_PATH` selon l'endroit où vous avez
uploadé le fichier (ex. dans Colab : `/content/NomDuFichier.zip`).


In [ ]:
# À adapter : chemin vers l'archive uploadée (ex. dans Colab après upload)
ZIP_PATH = "Basics_of_BERT_and_XLM-RoBERTa_-_PyTorch_-_2.zip"
EXTRACT_ROOT = Path("data_nli")

if not EXTRACT_ROOT.exists():
    EXTRACT_ROOT.mkdir(parents=True, exist_ok=True)
    with zipfile.ZipFile(ZIP_PATH, 'r') as zf:
        zf.extractall(EXTRACT_ROOT)
    print("Archive principale extraite.")

# Le zip principal contient un sous-dossier avec train.csv.zip / test.csv.zip imbriqués
inner_dir = next(EXTRACT_ROOT.glob("*"))  # dossier "Basics of BERT and XLM-RoBERTa - PyTorch"
print("Dossier interne:", inner_dir)

train_zip = inner_dir / "train.csv.zip"
test_zip  = inner_dir / "test.csv.zip"

train_extract_dir = inner_dir / "train_extract"
test_extract_dir  = inner_dir / "test_extract"

if not train_extract_dir.exists():
    with zipfile.ZipFile(train_zip, 'r') as zf:
        zf.extractall(train_extract_dir)
if not test_extract_dir.exists():
    with zipfile.ZipFile(test_zip, 'r') as zf:
        zf.extractall(test_extract_dir)

train_csv = next(train_extract_dir.glob("*.csv"))
test_csv  = next(test_extract_dir.glob("*.csv"))

df_train = pd.read_csv(train_csv)
df_test  = pd.read_csv(test_csv)

print("Train shape:", df_train.shape)
print("Test shape:", df_test.shape)
df_train.head()


In [ ]:
# Distribution des labels et des langues
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

label_names = {0: "entailment", 1: "neutral", 2: "contradiction"}
df_train["label_name"] = df_train["label"].map(label_names)

sns.countplot(data=df_train, x="label_name", ax=axes[0])
axes[0].set_title("Répartition des classes (NLI)")

top_langs = df_train["language"].value_counts().head(10)
sns.barplot(x=top_langs.values, y=top_langs.index, ax=axes[1])
axes[1].set_title("Top 10 langues (train)")

plt.tight_layout()
plt.show()

print(df_train["label"].value_counts(normalize=True))


**Observation attendue** : le jeu de données est multilingue (anglais, arabe, ourdou,
français, etc.) et globalement équilibré entre les 3 classes. Cette multiplicité de langues
est précisément ce qui justifie l'usage d'un transformateur multilingue pré-entraîné
(XLM-RoBERTa / DistilBERT multilingue) comme ligne de base en section 6.


## 2. Préparation des données

Pour rester rapide à exécuter, on travaille sur un **sous-échantillon** du jeu d'entraînement
(modifiable via `N_SAMPLES`). On construit :
- un split train/validation stratifié sur le label,
- un **vocabulaire maison** (word-level) pour l'encodeur personnalisé (sections 3-5),
- un **tokenizer pré-entraîné** (Hugging Face) pour la baseline (section 6).


In [ ]:
N_SAMPLES = 6000  # réduire/augmenter selon le temps de calcul disponible
MAX_LEN = 64

df_sub = df_train.sample(n=min(N_SAMPLES, len(df_train)), random_state=SEED).reset_index(drop=True)

df_tr, df_val = train_test_split(
    df_sub, test_size=0.15, stratify=df_sub["label"], random_state=SEED
)
print("Train:", len(df_tr), " Val:", len(df_val))


In [ ]:
# --- Vocabulaire maison (word-level, simple split sur espaces + ponctuation basique) ---
import re

def simple_tokenize(text):
    text = str(text).lower()
    text = re.sub(r"([.,!?;:'\"()])", r" \1 ", text)
    return text.split()

PAD, UNK, CLS, SEP = "<pad>", "<unk>", "<cls>", "<sep>"

def build_vocab(texts, min_freq=2, max_size=20000):
    freq = {}
    for t in texts:
        for tok in simple_tokenize(t):
            freq[tok] = freq.get(tok, 0) + 1
    vocab = [PAD, UNK, CLS, SEP] + [w for w, c in sorted(freq.items(), key=lambda x: -x[1])
                                     if c >= min_freq][:max_size]
    return {w: i for i, w in enumerate(vocab)}

all_texts = pd.concat([df_tr["premise"], df_tr["hypothesis"]])
vocab = build_vocab(all_texts)
VOCAB_SIZE = len(vocab)
print("Taille du vocabulaire:", VOCAB_SIZE)

def encode_pair(premise, hypothesis, vocab, max_len=MAX_LEN):
    toks = [CLS] + simple_tokenize(premise) + [SEP] + simple_tokenize(hypothesis) + [SEP]
    ids = [vocab.get(t, vocab[UNK]) for t in toks][:max_len]
    attn_mask = [1] * len(ids)
    while len(ids) < max_len:
        ids.append(vocab[PAD])
        attn_mask.append(0)
    return ids, attn_mask


In [ ]:
class NLIDatasetCustom(Dataset):
    def __init__(self, df, vocab, max_len=MAX_LEN):
        self.df = df.reset_index(drop=True)
        self.vocab = vocab
        self.max_len = max_len

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        row = self.df.iloc[idx]
        ids, mask = encode_pair(row["premise"], row["hypothesis"], self.vocab, self.max_len)
        return {
            "input_ids": torch.tensor(ids, dtype=torch.long),
            "attention_mask": torch.tensor(mask, dtype=torch.long),
            "label": torch.tensor(int(row["label"]), dtype=torch.long),
        }

train_ds_custom = NLIDatasetCustom(df_tr, vocab)
val_ds_custom   = NLIDatasetCustom(df_val, vocab)

BATCH_SIZE = 32
train_loader_custom = DataLoader(train_ds_custom, batch_size=BATCH_SIZE, shuffle=True)
val_loader_custom   = DataLoader(val_ds_custom, batch_size=BATCH_SIZE, shuffle=False)

batch = next(iter(train_loader_custom))
print({k: v.shape for k, v in batch.items()})


## 3. Implémentation de l'attention à une seule tête

### Objectif
Implémenter le bloc de base (Scaled Dot-Product Attention) avant de l'étendre à plusieurs
têtes.

### Explication
Pour chaque token, on calcule trois projections linéaires **Q** (query), **K** (key) et
**V** (value). Le score d'attention entre deux tokens est le produit scalaire `Q·Kᵀ`, mis à
l'échelle par `1/√d_k` (pour stabiliser les gradients quand `d_k` est grand), puis normalisé
par un `softmax` pour obtenir une distribution de probabilité sur les positions. La sortie est
la moyenne pondérée des `V` selon ces probabilités.


In [ ]:
class SingleHeadAttention(nn.Module):
    """Scaled Dot-Product Attention à une seule tête."""
    def __init__(self, hidden_dim, head_dim=None):
        super().__init__()
        head_dim = head_dim or hidden_dim
        self.head_dim = head_dim
        self.q_proj = nn.Linear(hidden_dim, head_dim)
        self.k_proj = nn.Linear(hidden_dim, head_dim)
        self.v_proj = nn.Linear(hidden_dim, head_dim)
        self.last_attn_weights = None  # pour inspection / visualisation

    def forward(self, x, mask=None):
        # x: (batch, seq_len, hidden_dim)
        Q = self.q_proj(x)  # (batch, seq_len, head_dim)
        K = self.k_proj(x)
        V = self.v_proj(x)

        scores = torch.matmul(Q, K.transpose(-2, -1)) / math.sqrt(self.head_dim)
        # scores: (batch, seq_len, seq_len)

        if mask is not None:
            # mask: (batch, seq_len) -> (batch, 1, seq_len), 0 = position à ignorer
            extended_mask = mask.unsqueeze(1)
            scores = scores.masked_fill(extended_mask == 0, float('-inf'))

        attn_weights = torch.softmax(scores, dim=-1)
        self.last_attn_weights = attn_weights.detach()  # log pour inspection

        output = torch.matmul(attn_weights, V)  # (batch, seq_len, head_dim)
        return output, attn_weights


In [ ]:
# Validation des formes avec des tenseurs factices
BATCH, SEQ_LEN, HIDDEN_DIM = 4, 10, 32

dummy_x = torch.randn(BATCH, SEQ_LEN, HIDDEN_DIM)
dummy_mask = torch.ones(BATCH, SEQ_LEN)
dummy_mask[:, 7:] = 0  # simulate padding

single_attn = SingleHeadAttention(hidden_dim=HIDDEN_DIM)
out, weights = single_attn(dummy_x, mask=dummy_mask)

print("Entrée :", dummy_x.shape)
print("Sortie  :", out.shape)              # attendu: (4, 10, 32)
print("Poids d'attention:", weights.shape)  # attendu: (4, 10, 10)
print("Somme des poids (doit valoir ~1 par ligne):", weights.sum(-1)[0])


## 4. Module d'attention multi-têtes

### Objectif
Étendre le bloc mono-tête à un mécanisme multi-têtes : on divise `hidden_dim` en `num_heads`
sous-espaces, on applique l'attention indépendamment dans chacun, puis on concatène et on
projette le résultat. Cela permet au modèle d'apprendre **plusieurs types de relations**
simultanément (ex. une tête peut capter des dépendances syntaxiques locales, une autre des
relations sémantiques à longue distance).

On ajoute également **Dropout** (régularisation) et une **connexion résiduelle** + 
`LayerNorm` (stabilité de l'entraînement, comme dans l'architecture Transformer originale).


In [ ]:
class MultiHeadAttention(nn.Module):
    def __init__(self, hidden_dim, num_heads, dropout=0.1):
        super().__init__()
        assert hidden_dim % num_heads == 0, "hidden_dim doit être divisible par num_heads"
        self.num_heads = num_heads
        self.head_dim = hidden_dim // num_heads
        self.hidden_dim = hidden_dim

        self.q_proj = nn.Linear(hidden_dim, hidden_dim)
        self.k_proj = nn.Linear(hidden_dim, hidden_dim)
        self.v_proj = nn.Linear(hidden_dim, hidden_dim)
        self.out_proj = nn.Linear(hidden_dim, hidden_dim)

        self.dropout = nn.Dropout(dropout)
        self.layer_norm = nn.LayerNorm(hidden_dim)

        self.last_attn_weights = None  # (batch, num_heads, seq_len, seq_len)

    def split_heads(self, x):
        # x: (batch, seq_len, hidden_dim) -> (batch, num_heads, seq_len, head_dim)
        batch, seq_len, _ = x.shape
        x = x.reshape(batch, seq_len, self.num_heads, self.head_dim)
        return x.permute(0, 2, 1, 3)

    def forward(self, x, mask=None):
        residual = x

        Q = self.split_heads(self.q_proj(x))  # (batch, heads, seq_len, head_dim)
        K = self.split_heads(self.k_proj(x))
        V = self.split_heads(self.v_proj(x))

        scores = torch.matmul(Q, K.transpose(-2, -1)) / math.sqrt(self.head_dim)
        # scores: (batch, heads, seq_len, seq_len)

        if mask is not None:
            extended_mask = mask.unsqueeze(1).unsqueeze(1)  # (batch, 1, 1, seq_len)
            scores = scores.masked_fill(extended_mask == 0, float('-inf'))

        attn_weights = torch.softmax(scores, dim=-1)
        attn_weights = self.dropout(attn_weights)
        self.last_attn_weights = attn_weights.detach()

        context = torch.matmul(attn_weights, V)  # (batch, heads, seq_len, head_dim)
        batch, heads, seq_len, head_dim = context.shape
        context = context.permute(0, 2, 1, 3).reshape(batch, seq_len, heads * head_dim)

        out = self.out_proj(context)
        out = self.dropout(out)
        out = self.layer_norm(out + residual)  # connexion résiduelle + normalisation
        return out


In [ ]:
# Exemple forward : formes d'entrée/sortie
NUM_HEADS = 4
mha = MultiHeadAttention(hidden_dim=HIDDEN_DIM, num_heads=NUM_HEADS, dropout=0.1)

out_mha = mha(dummy_x, mask=dummy_mask)
print("Entrée  :", dummy_x.shape)
print("Sortie  :", out_mha.shape)                       # attendu: (4, 10, 32) - identique à l'entrée
print("Poids d'attention (par tête):", mha.last_attn_weights.shape)  # (4, 4, 10, 10)


## 5. (Optionnel — implémenté) Pile d'encodeurs personnalisée et boucle d'entraînement

### Objectif
Construire un mini réseau de type "encodeur Transformer" (multi-head attention + feed-forward
+ LayerNorm) et l'entraîner sur la tâche NLI à 3 classes (premise + hypothesis concaténés).


In [ ]:
class FeedForward(nn.Module):
    def __init__(self, hidden_dim, ff_dim, dropout=0.1):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(hidden_dim, ff_dim),
            nn.GELU(),
            nn.Dropout(dropout),
            nn.Linear(ff_dim, hidden_dim),
        )
        self.dropout = nn.Dropout(dropout)
        self.layer_norm = nn.LayerNorm(hidden_dim)

    def forward(self, x):
        residual = x
        out = self.net(x)
        out = self.dropout(out)
        return self.layer_norm(out + residual)


class EncoderBlock(nn.Module):
    def __init__(self, hidden_dim, num_heads, ff_dim, dropout=0.1):
        super().__init__()
        self.mha = MultiHeadAttention(hidden_dim, num_heads, dropout)
        self.ff = FeedForward(hidden_dim, ff_dim, dropout)

    def forward(self, x, mask=None):
        x = self.mha(x, mask)
        x = self.ff(x)
        return x


class CustomEncoderClassifier(nn.Module):
    """Encodeur Transformer léger from scratch pour classification NLI à 3 classes."""
    def __init__(self, vocab_size, hidden_dim=128, num_heads=4, ff_dim=256,
                 num_layers=2, max_len=MAX_LEN, num_classes=3, dropout=0.1):
        super().__init__()
        self.token_emb = nn.Embedding(vocab_size, hidden_dim, padding_idx=0)
        self.pos_emb = nn.Embedding(max_len, hidden_dim)
        self.dropout = nn.Dropout(dropout)

        self.layers = nn.ModuleList([
            EncoderBlock(hidden_dim, num_heads, ff_dim, dropout) for _ in range(num_layers)
        ])

        self.classifier = nn.Sequential(
            nn.Linear(hidden_dim, hidden_dim),
            nn.GELU(),
            nn.Dropout(dropout),
            nn.Linear(hidden_dim, num_classes),
        )

    def forward(self, input_ids, attention_mask=None):
        batch, seq_len = input_ids.shape
        positions = torch.arange(seq_len, device=input_ids.device).unsqueeze(0).expand(batch, -1)

        x = self.token_emb(input_ids) + self.pos_emb(positions)
        x = self.dropout(x)

        for layer in self.layers:
            x = layer(x, attention_mask)

        # Pooling: on utilise la représentation du token [CLS] (position 0)
        cls_repr = x[:, 0, :]
        logits = self.classifier(cls_repr)
        return logits

    def get_last_attention(self):
        """Retourne les poids d'attention de la dernière couche (pour visualisation)."""
        return self.layers[-1].mha.last_attn_weights


In [ ]:
custom_model = CustomEncoderClassifier(
    vocab_size=VOCAB_SIZE, hidden_dim=128, num_heads=4, ff_dim=256,
    num_layers=2, max_len=MAX_LEN, num_classes=3, dropout=0.1
).to(DEVICE)

n_params = sum(p.numel() for p in custom_model.parameters())
print(f"Nombre de paramètres du modèle personnalisé : {n_params:,}")


In [ ]:
def train_custom_model(model, train_loader, val_loader, epochs=5, lr=3e-4):
    optimizer = torch.optim.AdamW(model.parameters(), lr=lr)
    criterion = nn.CrossEntropyLoss()
    history = {"train_loss": [], "val_loss": [], "val_acc": []}

    for epoch in range(epochs):
        model.train()
        total_loss = 0.0
        for batch in train_loader:
            input_ids = batch["input_ids"].to(DEVICE)
            attn_mask = batch["attention_mask"].to(DEVICE)
            labels = batch["label"].to(DEVICE)

            optimizer.zero_grad()
            logits = model(input_ids, attn_mask)
            loss = criterion(logits, labels)
            loss.backward()
            optimizer.step()
            total_loss += loss.item() * input_ids.size(0)

        train_loss = total_loss / len(train_loader.dataset)

        model.eval()
        val_loss, correct = 0.0, 0
        with torch.no_grad():
            for batch in val_loader:
                input_ids = batch["input_ids"].to(DEVICE)
                attn_mask = batch["attention_mask"].to(DEVICE)
                labels = batch["label"].to(DEVICE)

                logits = model(input_ids, attn_mask)
                loss = criterion(logits, labels)
                val_loss += loss.item() * input_ids.size(0)
                correct += (logits.argmax(-1) == labels).sum().item()

        val_loss /= len(val_loader.dataset)
        val_acc = correct / len(val_loader.dataset)

        history["train_loss"].append(train_loss)
        history["val_loss"].append(val_loss)
        history["val_acc"].append(val_acc)

        print(f"Époque {epoch+1}/{epochs} | train_loss={train_loss:.4f} "
              f"| val_loss={val_loss:.4f} | val_acc={val_acc:.4f}")

    return history

history_custom = train_custom_model(custom_model, train_loader_custom, val_loader_custom, epochs=5)


In [ ]:
plt.figure(figsize=(6, 4))
plt.plot(history_custom["train_loss"], label="train_loss")
plt.plot(history_custom["val_loss"], label="val_loss")
plt.xlabel("Époque"); plt.ylabel("Loss"); plt.legend()
plt.title("Encodeur personnalisé — courbes de perte")
plt.show()

print(f"Meilleure exactitude de validation (custom): {max(history_custom['val_acc']):.4f}")


## 6. Ligne de base : fine-tuning d'un Transformer pré-entraîné

On utilise **DistilBERT multilingue** (`distilbert-base-multilingual-cased`), adapté au jeu
de données NLI multilingue, comme ligne de base "grand modèle pré-entraîné" à comparer à
l'encodeur personnalisé léger. (Vous pouvez remplacer par `xlm-roberta-base` pour une
comparaison BERT vs XLM-RoBERTa, au prix d'un temps d'entraînement plus long.)


In [ ]:
from transformers import AutoTokenizer, AutoModelForSequenceClassification

PRETRAINED_NAME = "distilbert-base-multilingual-cased"

tokenizer = AutoTokenizer.from_pretrained(PRETRAINED_NAME)
pretrained_model = AutoModelForSequenceClassification.from_pretrained(
    PRETRAINED_NAME, num_labels=3
).to(DEVICE)

print("Tokenizer et modèle pré-entraîné chargés:", PRETRAINED_NAME)


In [ ]:
class NLIDatasetPretrained(Dataset):
    def __init__(self, df, tokenizer, max_len=MAX_LEN):
        self.df = df.reset_index(drop=True)
        self.tokenizer = tokenizer
        self.max_len = max_len

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        row = self.df.iloc[idx]
        enc = self.tokenizer(
            row["premise"], row["hypothesis"],
            truncation=True, padding="max_length", max_length=self.max_len,
            return_tensors="pt"
        )
        return {
            "input_ids": enc["input_ids"].squeeze(0),
            "attention_mask": enc["attention_mask"].squeeze(0),
            "label": torch.tensor(int(row["label"]), dtype=torch.long),
        }

train_ds_pt = NLIDatasetPretrained(df_tr, tokenizer)
val_ds_pt   = NLIDatasetPretrained(df_val, tokenizer)

train_loader_pt = DataLoader(train_ds_pt, batch_size=16, shuffle=True)
val_loader_pt   = DataLoader(val_ds_pt, batch_size=16, shuffle=False)


In [ ]:
def train_pretrained_model(model, train_loader, val_loader, epochs=2, lr=2e-5):
    optimizer = torch.optim.AdamW(model.parameters(), lr=lr)
    history = {"train_loss": [], "val_loss": [], "val_acc": []}

    for epoch in range(epochs):
        model.train()
        total_loss = 0.0
        for batch in train_loader:
            input_ids = batch["input_ids"].to(DEVICE)
            attn_mask = batch["attention_mask"].to(DEVICE)
            labels = batch["label"].to(DEVICE)

            optimizer.zero_grad()
            outputs = model(input_ids=input_ids, attention_mask=attn_mask, labels=labels)
            loss = outputs.loss
            loss.backward()
            optimizer.step()
            total_loss += loss.item() * input_ids.size(0)

        train_loss = total_loss / len(train_loader.dataset)

        model.eval()
        val_loss, correct = 0.0, 0
        with torch.no_grad():
            for batch in val_loader:
                input_ids = batch["input_ids"].to(DEVICE)
                attn_mask = batch["attention_mask"].to(DEVICE)
                labels = batch["label"].to(DEVICE)

                outputs = model(input_ids=input_ids, attention_mask=attn_mask, labels=labels)
                val_loss += outputs.loss.item() * input_ids.size(0)
                correct += (outputs.logits.argmax(-1) == labels).sum().item()

        val_loss /= len(val_loader.dataset)
        val_acc = correct / len(val_loader.dataset)

        history["train_loss"].append(train_loss)
        history["val_loss"].append(val_loss)
        history["val_acc"].append(val_acc)

        print(f"Époque {epoch+1}/{epochs} | train_loss={train_loss:.4f} "
              f"| val_loss={val_loss:.4f} | val_acc={val_acc:.4f}")

    return history

# 2 époques : suffisant pour une comparaison, augmenter si le temps/GPU le permettent
history_pretrained = train_pretrained_model(pretrained_model, train_loader_pt, val_loader_pt, epochs=2)


## 7. Comparaison encodeur personnalisé vs Transformer pré-entraîné

In [ ]:
n_params_pretrained = sum(p.numel() for p in pretrained_model.parameters())

comparison = pd.DataFrame({
    "Modèle": ["Encodeur personnalisé (from scratch)", f"{PRETRAINED_NAME} (fine-tuné)"],
    "Nb paramètres": [n_params, n_params_pretrained],
    "Meilleure val_acc": [max(history_custom["val_acc"]), max(history_pretrained["val_acc"])],
    "Dernière val_loss": [history_custom["val_loss"][-1], history_pretrained["val_loss"][-1]],
})
comparison


**Analyse attendue** : le modèle pré-entraîné dispose de très nombreux paramètres et de
connaissances linguistiques acquises sur un immense corpus multilingue ; il devrait obtenir
une exactitude de validation nettement supérieure, même avec peu d'époques de fine-tuning,
grâce à des représentations contextuelles déjà riches. L'encodeur personnalisé, entraîné
from scratch sur un petit sous-échantillon, est beaucoup plus rapide à entraîner et bien plus
léger, mais sa capacité à généraliser (notamment au changement de langue) est nettement plus
limitée faute de connaissances préalables.


## 8. Visualisation des cartes d'attention

On visualise les poids d'attention pour un exemple de paire (premise, hypothesis) :
- pour l'encodeur personnalisé (dernière couche, toutes les têtes),
- pour le modèle pré-entraîné (via `output_attentions=True`).


In [ ]:
def plot_attention_heads(attn_weights, tokens, title, max_heads=4):
    """attn_weights: (num_heads, seq_len, seq_len) - numpy array."""
    num_heads = min(attn_weights.shape[0], max_heads)
    fig, axes = plt.subplots(1, num_heads, figsize=(5 * num_heads, 5))
    if num_heads == 1:
        axes = [axes]
    for h in range(num_heads):
        sns.heatmap(attn_weights[h], xticklabels=tokens, yticklabels=tokens,
                    cmap='viridis', ax=axes[h], cbar=False)
        axes[h].set_title(f"Tête {h+1}")
        axes[h].tick_params(axis='x', rotation=90)
    fig.suptitle(title)
    plt.tight_layout()
    plt.show()


In [ ]:
# --- Visualisation pour l'encodeur personnalisé ---
sample_row = df_val.iloc[0]
ids, mask = encode_pair(sample_row["premise"], sample_row["hypothesis"], vocab, MAX_LEN)
input_ids_t = torch.tensor([ids]).to(DEVICE)
attn_mask_t = torch.tensor([mask]).to(DEVICE)

custom_model.eval()
with torch.no_grad():
    _ = custom_model(input_ids_t, attn_mask_t)

attn_custom = custom_model.get_last_attention()[0].cpu().numpy()  # (num_heads, seq_len, seq_len)
inv_vocab = {i: w for w, i in vocab.items()}
tokens_custom = [inv_vocab[i] for i in ids]
real_len = sum(mask)  # tronquer l'affichage au contenu réel (sans padding)

plot_attention_heads(
    attn_custom[:, :real_len, :real_len], tokens_custom[:real_len],
    "Encodeur personnalisé — dernière couche d'attention multi-têtes"
)


In [ ]:
# --- Visualisation pour le modèle pré-entraîné ---
enc = tokenizer(sample_row["premise"], sample_row["hypothesis"],
                 truncation=True, max_length=MAX_LEN, return_tensors="pt").to(DEVICE)

pretrained_model.eval()
with torch.no_grad():
    outputs = pretrained_model(**enc, output_attentions=True)

# attentions: tuple (num_layers) de tenseurs (batch, num_heads, seq_len, seq_len)
last_layer_attn = outputs.attentions[-1][0].cpu().numpy()  # (num_heads, seq_len, seq_len)
tokens_pretrained = tokenizer.convert_ids_to_tokens(enc["input_ids"][0].cpu())

plot_attention_heads(
    last_layer_attn, tokens_pretrained,
    f"{PRETRAINED_NAME} — dernière couche d'attention"
)


## 9. Réflexion comparative

### Attention mono-tête vs multi-têtes vs cross-attention

- **Attention mono-tête** : un seul jeu de projections Q/K/V apprend **un seul type** de
  relation entre tokens (ex. uniquement la proximité syntaxique, ou uniquement la
  co-référence). C'est une représentation limitée : si une relation pertinente n'est pas
  capturée par cet unique sous-espace, le modèle ne peut pas la représenter.
- **Attention multi-têtes** : en exécutant plusieurs attentions en parallèle dans des
  sous-espaces différents (`num_heads` projections indépendantes), le modèle peut capter
  **simultanément plusieurs types de relations** (syntaxique, sémantique, position relative,
  etc.), puis les combine par concaténation + projection linéaire. C'est strictement plus
  expressif qu'une seule tête, pour un coût de calcul similaire (la dimension totale est
  répartie entre les têtes).
- **Cross-attention** : contrairement à l'auto-attention (Q, K, V dérivés de la même
  séquence), la cross-attention calcule Q à partir d'une séquence et K/V à partir d'une
  **autre** séquence (ex. dans un encodeur-décodeur : Q vient du décodeur, K/V de l'encodeur).
  Elle permet à une séquence de "regarder" une autre séquence — utile par exemple pour aligner
  l'hypothesis sur la premise token par token plutôt que de les concaténer comme on l'a fait
  ici.

### Compromis pile personnalisée légère vs grand modèle pré-entraîné

| Critère | Encodeur personnalisé (from scratch) | Transformer pré-entraîné (fine-tuné) |
|---|---|---|
| Nombre de paramètres | Très faible (~centaines de K) | Très élevé (dizaines à centaines de millions) |
| Temps d'entraînement | Rapide, peu de ressources | Plus lent, GPU recommandé |
| Connaissances préalables | Aucune (apprend tout sur le petit jeu de données) | Connaissances linguistiques riches issues du pré-entraînement |
| Généralisation multilingue | Faible (vocabulaire et structure appris sur l'échantillon réduit) | Forte (pré-entraîné sur de nombreuses langues) |
| Contrôle / interprétabilité | Total (architecture entièrement maîtrisée) | Partiel (architecture figée, fine-tuning limité) |
| Cas d'usage typique | Prototypage rapide, contraintes mémoire fortes, pédagogie | Production, exigence de performance/robustesse |

**Conclusion** : pour une tâche de NLI multilingue avec peu de données d'entraînement par
langue, le fine-tuning d'un Transformer pré-entraîné est presque toujours préférable en
pratique. L'implémentation d'un encodeur personnalisé reste néanmoins très utile sur le plan
pédagogique pour comprendre précisément le fonctionnement interne du mécanisme d'attention,
et peut s'avérer pertinente dans des contextes à très fortes contraintes de ressources
(edge computing, latence stricte) où un modèle pré-entraîné serait trop coûteux.


## 10. Liste de contrôle des livrables

- [x] Module `SingleHeadAttention` implémenté et validé sur tenseurs factices (section 3)
- [x] Module `MultiHeadAttention` avec dropout + connexion résiduelle, exemple forward
      (section 4)
- [x] Pile d'encodeurs personnalisée (`CustomEncoderClassifier`) + boucle d'entraînement sur
      le jeu de données NLI (section 5)
- [x] Ligne de base Transformer pré-entraîné fine-tunée sur le même jeu de données
      (section 6)
- [x] Comparaison chiffrée des deux approches (section 7)
- [x] Visualisations des cartes d'attention pour les deux modèles (section 8)
- [x] Réflexion écrite comparant mono-tête / multi-têtes / cross-attention et les compromis
      pile légère vs grand modèle pré-entraîné (section 9)

### Rappels rapides
- `N_SAMPLES`, `MAX_LEN`, `epochs` sont volontairement réduits pour un temps d'exécution
  raisonnable ; augmentez-les si vous disposez de plus de temps/ressources GPU.
- Le split train/validation est stratifié et fixé par `SEED` pour comparer les expériences
  de façon cohérente.
- En cas de résultat surprenant, vérifiez en priorité l'encodage des paires premise/hypothesis
  (troncature, padding, alignement du masque) avant de remettre en cause les modules
  d'attention eux-mêmes.
